In [1]:
# Importations
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
import mlflow
import mlflow.pytorch

C:\Users\USER\anaconda3\envs\ecg_project\lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


In [2]:
# -----------------------------
# Paramètres
# -----------------------------
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Résolution robuste du dossier processed (remonte l'arborescence depuis le cwd)
def find_processed_dir():
    cur = os.getcwd()
    while True:
        candidate = os.path.join(cur, "data", "processed")
        if os.path.isdir(candidate):
            return os.path.abspath(candidate)
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    # fallback: try relative to this notebook file location if possible
    # (Jupyter notebooks may start with different working dirs)
    possible = os.path.abspath(os.path.join('..', '..', 'data', 'processed'))
    if os.path.isdir(possible):
        return possible
    raise FileNotFoundError(f'Could not find data/processed starting from cwd={os.getcwd()}')

PROCESSED_DIR = find_processed_dir()
NUM_CLASSES = 2
IMAGE_SIZE = 224

In [3]:
# -----------------------------
# Chargement des données
# -----------------------------
images = np.load(os.path.join(PROCESSED_DIR, "images.npy"))  # (N,224,224,1)
labels = np.load(os.path.join(PROCESSED_DIR, "labels.npy"))

# Corriger dimensions
images = images.squeeze(-1)  # (N,224,224)

X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)  # (N,1,224,224)
y = torch.tensor(labels, dtype=torch.long)

dataset = TensorDataset(X, y)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batch:", next(iter(train_loader))[0].shape)

Train batch: torch.Size([16, 1, 224, 224])


In [4]:
# -----------------------------
# Modèle ResNet18 (ImageNet)
# -----------------------------
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Adapter la première couche pour grayscale
old_conv = model.conv1
model.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=False
)

with torch.no_grad():
    model.conv1.weight[:, 0, :, :] = old_conv.weight.mean(dim=1)

# Adapter la dernière couche (2 classes)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(DEVICE)


In [5]:
# -----------------------------
# Loss & Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [6]:
# -----------------------------
# MLflow
# -----------------------------
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("ResNet_ECG")

if mlflow.active_run():
    mlflow.end_run()

2026/02/02 22:05:46 INFO mlflow.tracking.fluent: Experiment with name 'ResNet_ECG' does not exist. Creating a new experiment.


In [7]:
# -----------------------------
# Entraînement
# -----------------------------
with mlflow.start_run(run_name="ResNet18_ECG"):

    mlflow.log_param("model", "ResNet18")
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)

    for epoch in range(EPOCHS):
        # ===== Train =====
        model.train()
        train_loss, train_correct = 0.0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            train_correct += (outputs.argmax(1) == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_acc = train_correct / len(train_loader.dataset)

        # ===== Validation =====
        model.eval()
        val_loss, val_correct = 0.0, 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                val_correct += (outputs.argmax(1) == targets).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)

        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)

    # -----------------------------
    # Sauvegarde
    # -----------------------------
    mlflow.pytorch.log_model(model, "resnet18_ecg")

print("✅ Entraînement ResNet terminé avec succès")

Epoch [1/15] Train Loss: 0.3392 | Train Acc: 0.8838 Val Loss: 8.1131 | Val Acc: 0.5288


Epoch [2/15] Train Loss: 0.0893 | Train Acc: 0.9661 Val Loss: 0.2952 | Val Acc: 0.9038


Epoch [3/15] Train Loss: 0.0856 | Train Acc: 0.9734 Val Loss: 0.1752 | Val Acc: 0.9423


Epoch [4/15] Train Loss: 0.0356 | Train Acc: 0.9879 Val Loss: 0.0310 | Val Acc: 0.9904


Epoch [5/15] Train Loss: 0.0378 | Train Acc: 0.9879 Val Loss: 0.0129 | Val Acc: 1.0000


...

Epoch [12/15] Train Loss: 0.0125 | Train Acc: 0.9952 Val Loss: 0.7090 | Val Acc: 0.8077


Epoch [13/15] Train Loss: 0.0548 | Train Acc: 0.9806 Val Loss: 0.0373 | Val Acc: 0.9808


Epoch [14/15] Train Loss: 0.0710 | Train Acc: 0.9758 Val Loss: 2.0874 | Val Acc: 0.5481


Epoch [15/15] Train Loss: 0.0840 | Train Acc: 0.9685 Val Loss: 0.4574 | Val Acc: 0.8942


✅ Entraînement ResNet terminé avec succès
